# 3DHandReconstruction — 3D hand localization & reconstruction on Amazon SageMaker

This notebook takes the 3DHandReconstruction model package from your AWS Marketplace subscription through the complete workflow:

1. deploy a real-time endpoint,
2. run **detection** on the bundled sample images (no data of your own is needed),
3. run **3D reconstruction** — this step needs your own licensed MANO file and is skipped automatically if you don't have one yet,
4. run a **batch transform** job,
5. clean up.

It uses only `boto3`. Requirements: `pip install -r requirements.txt`.

**Before you start**
- Subscribe to 3DHandReconstruction on AWS Marketplace and copy the **model package ARN** for your region from the subscription's *Configuration* page.
- Have an IAM role that SageMaker can assume with S3 access to your bucket (inside SageMaker Studio or a notebook instance, the notebook's own role is detected automatically).
- Optional: a converted MANO file `mano_right.npz` next to this notebook (see the README for the one-time conversion and licensing).

In [ ]:
# ---------------- Parameters: edit these ----------------
MODEL_PACKAGE_ARN = ""                    # from your AWS Marketplace subscription (Configuration -> model package ARN)
ENDPOINT_NAME = "hand-reconstruction"
INSTANCE_TYPE = "ml.g4dn.xlarge"          # recommended (GPU, FP16). CPU alternative: "ml.m5.xlarge"
BATCH_INSTANCE_TYPE = "ml.m5.xlarge"
ROLE_ARN = ""                             # SageMaker execution role; leave empty inside SageMaker Studio / notebook instances
S3_BUCKET = ""                            # for batch input/output; empty = sagemaker-<region>-<account> (created if missing)
MANO_NPZ = "mano_right.npz"               # optional converted MANO file; 3D sections are skipped if it is absent

In [ ]:
import base64, json, os, time, uuid
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from make_batch_input import build as build_batch_input
from hand_client import HandClient

session = boto3.session.Session()
region = session.region_name
account = session.client("sts").get_caller_identity()["Account"]
sm = session.client("sagemaker")
s3 = session.client("s3")
print(f"region={region} account={account}")

assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), "Set MODEL_PACKAGE_ARN from your Marketplace subscription"
assert MODEL_PACKAGE_ARN.split(":")[3] == region, "Use the model package ARN for this region"

if not ROLE_ARN:  # inside SageMaker, the caller is the notebook's execution role
    caller = session.client("sts").get_caller_identity()["Arn"]
    if ":assumed-role/" in caller:
        ROLE_ARN = f"arn:aws:iam::{account}:role/{caller.split('/')[1]}"
    else:
        raise RuntimeError("Set ROLE_ARN to a SageMaker execution role")
if not S3_BUCKET:
    S3_BUCKET = f"sagemaker-{region}-{account}"
    try:
        s3.head_bucket(Bucket=S3_BUCKET)
    except s3.exceptions.ClientError:
        kwargs = {} if region == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": region}}
        s3.create_bucket(Bucket=S3_BUCKET, **kwargs)
SAMPLE_IMAGES = sorted(p for p in Path("samples/images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
Path("outputs").mkdir(exist_ok=True)
print(f"role={ROLE_ARN}\nbucket={S3_BUCKET}\n{len(SAMPLE_IMAGES)} sample images")

## 1. Deploy a real-time endpoint

Marketplace model packages run with network isolation; the model is created directly from the package ARN.

In [ ]:
MODEL_NAME = ENDPOINT_NAME + "-model"
CONFIG_NAME = ENDPOINT_NAME + "-config"

sm.create_model(ModelName=MODEL_NAME, ExecutionRoleArn=ROLE_ARN, EnableNetworkIsolation=True,
                PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN})
sm.create_endpoint_config(EndpointConfigName=CONFIG_NAME, ProductionVariants=[{
    "VariantName": "primary", "ModelName": MODEL_NAME,
    "InstanceType": INSTANCE_TYPE, "InitialInstanceCount": 1}])
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)

print("creating endpoint (typically 5-10 minutes) ", end="", flush=True)
while True:
    status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
    if status == "InService":
        break
    if status == "Failed":
        raise RuntimeError(sm.describe_endpoint(EndpointName=ENDPOINT_NAME).get("FailureReason"))
    print(".", end="", flush=True); time.sleep(30)
print(" InService")

## 2. Detection — works out of the box

Bounding boxes, left/right handedness and 21 2D keypoints per hand. No MANO file is needed for this mode.

In [ ]:
client = HandClient(ENDPOINT_NAME, MANO_NPZ if Path(MANO_NPZ).exists() else None, boto_session=session)

detections = {}
for img in SAMPLE_IMAGES[:3]:
    t0 = time.time()
    detections[img] = client.predict(img, detect_only=True)
    hands = detections[img]["hands"]
    print(f"{img.name}: {len(hands)} hands in {time.time() - t0:.2f}s "
          f"({sum(h['is_right'] for h in hands)} right, {sum(1 - h['is_right'] for h in hands)} left)")

def show_detections(img_path, result, ax):
    ax.imshow(Image.open(img_path)); ax.set_axis_off()
    for h in result["hands"]:
        color = "lime" if h["is_right"] else "red"
        x0, y0, x1, y1 = h["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=color, lw=2))
        ax.text(x0, y0 - 4, "R" if h["is_right"] else "L", color=color, fontsize=11, weight="bold")
        pts = np.asarray(h["keypoints_2d_det"])
        ax.scatter(pts[:, 0], pts[:, 1], s=10, c=color)

fig, axes = plt.subplots(1, len(detections), figsize=(6 * len(detections), 6))
for ax, (img, res) in zip(np.atleast_1d(axes), detections.items()):
    show_detections(img, res, ax); ax.set_title(img.name)
fig.savefig("outputs/detections.png", bbox_inches="tight"); plt.show()

## 3. 3D reconstruction — requires your MANO file

If `mano_right.npz` is present, the client attaches it once (the endpoint caches it in memory by SHA-256 and answers `MANO_REQUIRED` whenever a fresh instance needs it again — the client handles that retry for you). Each hand then comes back with MANO parameters, 21 3D joints and a 778-vertex mesh.

If the file is absent this section prints a note and is skipped — the rest of the notebook still runs.

In [ ]:
HAVE_MANO = Path(MANO_NPZ).exists()
if not HAVE_MANO:
    print(f"{MANO_NPZ} not found — skipping 3D reconstruction. See README: MANO model, bring your own.")
else:
    img = SAMPLE_IMAGES[0]
    t0 = time.time()
    recon = client.predict(img)  # full reconstruction; MANO attached automatically if needed
    print(f"{img.name}: {len(recon['hands'])} hands reconstructed in {time.time() - t0:.2f}s; MANO sha256 {recon['mano_sha256'][:12]}...")
    hand = recon["hands"][0]
    print("per-hand fields:", {k: (np.asarray(v).shape if isinstance(v, list) else v) for k, v in hand.items() if k != "mano_params"})
    print("mano_params:", {k: np.asarray(v).shape for k, v in hand["mano_params"].items()})

    fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(Image.open(img)); ax.set_axis_off()
    for h in recon["hands"]:
        pts = np.asarray(h["keypoints_2d"])           # the 778 mesh vertices projected into the image
        ax.scatter(pts[:, 0], pts[:, 1], s=1, alpha=0.5, c="lime" if h["is_right"] else "red")
    ax.set_title("projected mesh vertices (green = right, red = left)")
    fig.savefig("outputs/reconstruction_2d.png", bbox_inches="tight"); plt.show()
    json.dump(recon, open("outputs/reconstruction.json", "w"))

In [ ]:
# Optional: render the first reconstructed mesh headless (pip install pyrender trimesh). Mesh faces come from YOUR mano_right.npz.
if HAVE_MANO:
    try:
        os.environ["PYOPENGL_PLATFORM"] = "egl"
        import pyrender, trimesh
        hand = recon["hands"][0]
        faces = np.load(MANO_NPZ)["f"]
        if not hand["is_right"]:
            faces = faces[:, ::-1]                    # flip winding for left hands
        mesh = trimesh.Trimesh(np.asarray(hand["vertices"]), faces)
        mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi, [1, 0, 0]))  # OpenCV -> OpenGL convention
        mesh.apply_translation(-mesh.centroid)
        scene = pyrender.Scene(bg_color=[1.0, 1.0, 1.0], ambient_light=[0.3, 0.3, 0.3])
        scene.add(pyrender.Mesh.from_trimesh(mesh, smooth=True))
        cam_pose = np.eye(4); cam_pose[2, 3] = 0.35
        scene.add(pyrender.PerspectiveCamera(yfov=0.8), pose=cam_pose)
        scene.add(pyrender.DirectionalLight(intensity=3.0), pose=cam_pose)
        color, _ = pyrender.OffscreenRenderer(600, 600).render(scene)
        plt.figure(figsize=(5, 5)); plt.imshow(color); plt.axis("off"); plt.show()
    except ImportError:
        print("pyrender/trimesh not installed — skipping the 3D render (pip install pyrender trimesh)")

## 4. Batch transform

For large image sets. `make_batch_input.py` writes one self-contained request per line; this run is detection-only so it needs no MANO file (for 3D jobs pass `mano_npz=` and every line carries the file — lines are processed independently).

In [ ]:
job = f"hand-batch-{uuid.uuid4().hex[:8]}"
n = build_batch_input(SAMPLE_IMAGES[:3], "batch_input.jsonl", options={"detect_only": True})
s3.upload_file("batch_input.jsonl", S3_BUCKET, f"hand-reconstruction/{job}/input/batch_input.jsonl")

sm.create_transform_job(
    TransformJobName=job, ModelName=MODEL_NAME, MaxPayloadInMB=10, BatchStrategy="SingleRecord",
    TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix",
                                                     "S3Uri": f"s3://{S3_BUCKET}/hand-reconstruction/{job}/input/"}},
                    "ContentType": "application/json", "SplitType": "Line"},
    TransformOutput={"S3OutputPath": f"s3://{S3_BUCKET}/hand-reconstruction/{job}/output/",
                     "Accept": "application/json", "AssembleWith": "Line"},
    TransformResources={"InstanceType": BATCH_INSTANCE_TYPE, "InstanceCount": 1})
print(f"transform job {job} ({n} images) ", end="", flush=True)
while True:
    status = sm.describe_transform_job(TransformJobName=job)["TransformJobStatus"]
    if status == "Completed":
        break
    if status in ("Failed", "Stopped"):
        raise RuntimeError(sm.describe_transform_job(TransformJobName=job).get("FailureReason"))
    print(".", end="", flush=True); time.sleep(30)
print(" Completed")

Path("batch_output").mkdir(exist_ok=True)
s3.download_file(S3_BUCKET, f"hand-reconstruction/{job}/output/batch_input.jsonl.out", "batch_output/batch_input.jsonl.out")
for img, line in zip(SAMPLE_IMAGES[:3], open("batch_output/batch_input.jsonl.out")):
    r = json.loads(line)
    print(f"{img.name}: {r.get('error') or str(len(r['hands'])) + ' hands'}")

## 5. Clean up

Delete the endpoint to stop the hourly charge (the model and config are removed too).

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=CONFIG_NAME)
sm.delete_model(ModelName=MODEL_NAME)
print("deleted endpoint, endpoint config and model")

## Notes

- **`MANO_REQUIRED` in batch output** means a line was missing `mano_npz`; build 3D inputs with `make_batch_input.py --mano_npz ...`.
- **Options**: `conf` (detector threshold, default 0.3), `iou` (NMS, 0.7), `rescale_factor` (crop padding, 2.0), `mesh` (`false` drops the 778 vertices), `detect_only`.
- **Precision**: GPU instances run FP16 (sub-millimetre differences vs CPU FP32); CPU instances run FP32.
- **Limits**: 25 MB per request, 60 s per invocation. Downscale very large photos first.
- Full request/response reference: [README.md](README.md).